In [1]:
# library
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset
import multiprocessing
from sklearn.model_selection import train_test_split
from datasets import Dataset

from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

from google.colab import drive
drive.mount('/content/drive')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# downloaded package
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 155.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.1/415.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
# model hyperparameters
CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 1e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 50,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "eval_steps": 500,
    "save_steps": 500,
    "output_dir": "/content/outputs",
}

In [3]:
SYSTEM_PROMPT = (
    "You are an SVG generation model.\n"
    "Given a user prompt, generate exactly one valid SVG.\n"
    "Strict Requirements:\n"
    "- The SVG canvas must be 256x256 with viewBox=\"0 0 256 256\"\n"
    "- Use only valid SVG elements such as svg, g, path, rect, circle, ellipse, line, polyline, polygon\n"
    "- Do not include scripts, animation, foreignObject, event handlers, or external references\n"
    "- The SVG must be valid parseable XML\n"
    "- The total SVG length must be under 16000 characters\n"
    "- The number of path elements must not exceed 256\n"
    "- The output must start with <svg xmlns=\"http://www.w3.org/2000/svg\" width=\"256\" height=\"256\" viewBox=\"0 0 256 256\"> and end with </svg>\n"
    "Do not include any explanation, markdown, or extra text.\n"
    "Return only the SVG string."
)

STANDARD_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'

DISALLOWED_TAGS = {
    "script", "animate", "animatemotion", "animatetransform",
    "set", "foreignobject"
}

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line",
    "polyline", "polygon", "defs", "use", "symbol", "clippath",
    "mask", "lineargradient", "radialgradient", "stop", "text",
    "tspan", "title", "desc", "style", "pattern", "marker", "filter"
}


def normalize_svg(svg_text: str):
    svg_text = str(svg_text).strip()

    svg_text = re.sub(r"^\s*<\?xml[^>]*\?>", "", svg_text).strip()

    if not svg_text.startswith("<svg"):
        return None

    svg_text = re.sub(r"^<svg\b[^>]*>", STANDARD_HEADER, svg_text, count=1)

    if len(svg_text) > 16000:
        return None

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return None

    root_tag = root.tag.split("}")[-1].lower()
    if root_tag != "svg":
        return None

    path_count = 0
    for elem in root.iter():
        tag = elem.tag.split("}")[-1].lower()

        if tag in DISALLOWED_TAGS:
            return None

        if tag not in ALLOWED_TAGS:
            return None

        if tag == "path":
            path_count += 1

        for attr in elem.attrib:
            if attr.lower().startswith("on"):
                return None

        for attr_val in elem.attrib.values():
            val = str(attr_val).lower()
            if "http://" in val or "https://" in val:
                return None

    if path_count > 256:
        return None

    return svg_text


def format_sft_text(example):
    user_prompt = str(example["prompt"]).strip()
    clean_svg = normalize_svg(example["svg"])

    if clean_svg is None:
        return {"text": None}

    return {
        "text": f"Generate an SVG of {user_prompt}\n{clean_svg}"
    }

In [4]:
# train/val dataset

df = pd.read_csv("/content/drive/MyDrive/dl-spring-2026-svg-generation/train.csv")

val_df = df.sample(n=500, random_state=42)

train_pool = df.drop(val_df.index)

train_df = train_pool.sample(n=12000, random_state=42).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
eval_ds = Dataset.from_pandas(val_df, preserve_index=False)

dataset_num_proc = min(multiprocessing.cpu_count(), 2)

train_text = train_ds.map(
    format_sft_text,
    remove_columns=train_ds.column_names,
    num_proc=dataset_num_proc
)
eval_text = eval_ds.map(
    format_sft_text,
    remove_columns=eval_ds.column_names,
    num_proc=dataset_num_proc
)

train_text = train_text.filter(lambda x: x["text"] is not None)
eval_text = eval_text.filter(lambda x: x["text"] is not None)

print("Train size after filtering:", len(train_text))
print("Eval size after filtering:", len(eval_text))
print(train_text[0]["text"][:500])

Map (num_proc=2):   0%|          | 0/12000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Train size after filtering: 11996
Eval size after filtering: 500
Generate an SVG of The image features a black outline of a rectangular icon with rounded corners, containing a curved arrow symbol in the center.
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><path fill="#231815" fill-opacity="1.0"  filling="0" d="M209.8308563232422 200.0 L35.0703125 200.0 A35.11172103881836 35.11172103881836 0.0 0 1 0.0 164.9267578125 L0.0 57.49589920043945 A35.11172103881836 35.11172103881836 0.0 0 1 35.0703125 22.42266082763672 L51.696


In [5]:
# set the model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Unsloth 2026.3.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
# start training

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps = CONFIG.get("warmup_steps", 50),
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=1,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_text,
    eval_dataset = eval_text,
    dataset_text_field = "text",
    max_seq_length = CONFIG["max_seq_length"],
    packing = False,
    dataset_num_proc = 2,
    args = training_args,
)

train_result = trainer.train()

model.save_pretrained("/content/lora_model_newstart_0327")
tokenizer.save_pretrained("/content/lora_model_newstart_0327")

!cp -r /content/lora_model_newstart0327 /content/drive/MyDrive/

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/11996 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,996 | Num Epochs = 3 | Total steps = 4,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 